# Fallstudie: Energieverbrauchsanalyse

**Szenario**: Du bist Analyst in der Nachhaltigkeitsabteilung eines produzierenden Unternehmens. Du analysierst den stündlichen Stromverbrauch über ein ganzes Jahr. Ziel: Einsparpotenziale identifizieren, Ausreißer bereinigen und Muster in der Zeitreihe verstehen.

**Besonderheiten dieses Datensatzes:**
- Zeitreihendaten: 8.760 Messpunkte (stündlich, ganzes Jahr)
- Fehlende Werte durch Sensor-Ausfälle (~5%)
- Saisonale Muster (Sommer vs. Winter)
- Schicht-Struktur (Früh/Spät/Nacht)

**DAV-Schwerpunkte:** Zeitreihen-Bereinigung, Interpolation, Rolling Windows, Ausreißer-Erkennung

---

In [ ]:
# ── Setup: Pakete installieren (nur in Colab nötig) ──────────────────────────
import subprocess, sys
packages = ['ydata-profiling', 'missingno', 'plotly', 'kaleido']
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)
print("Setup abgeschlossen!")

In [ ]:
# ── Daten laden von GitHub ────────────────────────────────────────────────────
BASE_URL = "https://raw.githubusercontent.com/swrobuts/dav/main/data/"
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
print("Bibliotheken geladen!")

## Phase 1: Business Understanding

**Analyseziele:**

1. In welchen Stunden und Schichten ist der Verbrauch am höchsten?
2. Wie stark schwankt der Verbrauch saisonal?
3. Wo gibt es Ausreißer (defekte Sensoren, untypische Ereignisse)?
4. Wie viel Energie könnte durch Optimierung der Nachtschicht gespart werden?

**Technische Herausforderungen:**
- Fehlende Werte müssen sinnvoll interpoliert werden (nicht einfach mit 0 füllen!)
- Zeitstempel müssen korrekt als datetime verarbeitet werden
- Ausreißer müssen identifiziert, aber nicht zwingend entfernt werden

In [ ]:
# ── Daten laden: Timestamp als datetime parsen ────────────────────────────────
df = pd.read_csv(BASE_URL + "energie_verbrauch.csv", parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"Datensatz geladen: {df.shape[0]:,} Messwerte, {df.shape[1]} Spalten")
print(f"Zeitraum: {df['timestamp'].min()} bis {df['timestamp'].max()}")
print(f"Auflösung: stündlich ({df['timestamp'].diff().mode()[0]})")
print()
df.head(10)

In [ ]:
# ── Zeitreihe überblicken ─────────────────────────────────────────────────────
print("=== Statistische Überblick ===")
print(df[['verbrauch_kwh', 'temperatur']].describe().round(2))
print()
print(f"Fehlende Messwerte (verbrauch_kwh): {df['verbrauch_kwh'].isna().sum()} ({df['verbrauch_kwh'].isna().mean()*100:.1f}%)")
print(f"Schichten: {df['schicht'].value_counts().to_dict()}")

# Schnellübersicht als Zeitreihe (erste 30 Tage)
df_jan = df[df['timestamp'] < '2024-02-01']
fig = px.line(
    df_jan, x='timestamp', y='verbrauch_kwh',
    title='Energieverbrauch Januar 2024 (stündlich)',
    labels={'timestamp': 'Zeit', 'verbrauch_kwh': 'Verbrauch (kWh)'},
    height=350,
)
fig.update_traces(line_width=0.8)
fig.show()

In [ ]:
# ── Fehlende Werte analysieren ────────────────────────────────────────────────
import missingno as msno
import matplotlib.pyplot as plt

print("=== Fehlende Werte pro Spalte ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(pd.DataFrame({'Fehlend': missing, 'Anteil (%)': missing_pct}))

# Missingno-Matrix (für die ersten 500 Zeilen als Überblick)
fig_msno, ax = plt.subplots(figsize=(10, 3))
msno.matrix(df.head(500), ax=ax, sparkline=False)
ax.set_title('Fehlende Werte – erste 500 Zeilen')
plt.tight_layout()
plt.show()

# Verteilung der Fehlwerte nach Monat
df['nan_verbrauch'] = df['verbrauch_kwh'].isna().astype(int)
nan_monat = df.groupby('monat')['nan_verbrauch'].sum().reset_index()
fig = px.bar(nan_monat, x='monat', y='nan_verbrauch',
             title='Fehlende Messwerte nach Monat',
             labels={'monat': 'Monat', 'nan_verbrauch': 'Anzahl fehlender Werte'},
             color='nan_verbrauch', color_continuous_scale='Oranges')
fig.show()

In [ ]:
# ── Fehlende Werte bereinigen: Lineare Interpolation ─────────────────────────
# Warum lineare Interpolation?
# Energieverbrauch ändert sich graduell – ein Mittelwert zwischen Nachbar-Werten ist sinnvoll.
# NICHT sinnvoll: forward-fill (würde Trends ignorieren) oder Nullen einsetzen.

df['verbrauch_bereinigt'] = df['verbrauch_kwh'].interpolate(method='linear')

# Vergleich: original vs. bereinigt
verbleibend = df['verbrauch_bereinigt'].isna().sum()
print(f"Fehlende Werte vor Interpolation: {df['verbrauch_kwh'].isna().sum()}")
print(f"Fehlende Werte nach Interpolation: {verbleibend}")
print()

# Visualisierung: Interpolierte Lücke
# Zeige einen Bereich mit Lücken
mask_nan = df['verbrauch_kwh'].isna()
first_nan_idx = mask_nan[mask_nan].index[0]
df_zoom = df.iloc[max(0, first_nan_idx - 6):first_nan_idx + 12].copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=df_zoom['timestamp'], y=df_zoom['verbrauch_kwh'],
                         mode='markers+lines', name='Original (mit Lücken)',
                         marker_color='salmon', line_dash='dot'))
fig.add_trace(go.Scatter(x=df_zoom['timestamp'], y=df_zoom['verbrauch_bereinigt'],
                         mode='lines', name='Interpoliert',
                         marker_color='steelblue'))
fig.update_layout(title='Vergleich: Original vs. Interpoliert (Ausschnitt)',
                  xaxis_title='Zeit', yaxis_title='Verbrauch (kWh)', height=350)
fig.show()

In [ ]:
# ── Monatlicher Durchschnittsverbrauch ────────────────────────────────────────
monat_stats = df.groupby('monat').agg(
    verbrauch_mean=('verbrauch_bereinigt', 'mean'),
    verbrauch_sum=('verbrauch_bereinigt', 'sum'),
    temperatur_mean=('temperatur', 'mean')
).reset_index().round(2)

monat_namen = {1:'Jan', 2:'Feb', 3:'Mär', 4:'Apr', 5:'Mai', 6:'Jun',
               7:'Jul', 8:'Aug', 9:'Sep', 10:'Okt', 11:'Nov', 12:'Dez'}
monat_stats['monat_name'] = monat_stats['monat'].map(monat_namen)

fig = make_subplots(rows=2, cols=1,
    subplot_titles=['Durchschnittlicher stündlicher Verbrauch (kWh) nach Monat',
                    'Durchschnittstemperatur nach Monat (°C)'],
    shared_xaxes=True)

fig.add_trace(
    go.Scatter(x=monat_stats['monat_name'], y=monat_stats['verbrauch_mean'],
               mode='lines+markers', name='Ø Verbrauch',
               line=dict(color='steelblue', width=2), marker_size=8),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=monat_stats['monat_name'], y=monat_stats['temperatur_mean'],
               mode='lines+markers', name='Ø Temperatur',
               line=dict(color='coral', width=2), marker_size=8),
    row=2, col=1
)
fig.update_layout(title='Saisonaler Verlauf: Verbrauch und Temperatur', height=500)
fig.show()

In [ ]:
# ── Heatmap: Verbrauch nach Stunde × Wochentag ───────────────────────────────
df['wochentag_num'] = pd.to_datetime(df['timestamp']).dt.dayofweek
wochentag_map = {0:'Mo', 1:'Di', 2:'Mi', 3:'Do', 4:'Fr', 5:'Sa', 6:'So'}
df['wochentag_kurz'] = df['wochentag_num'].map(wochentag_map)

pivot_heat = (df.groupby(['stunde', 'wochentag_kurz'])['verbrauch_bereinigt']
               .mean()
               .reset_index()
               .pivot(index='stunde', columns='wochentag_kurz', values='verbrauch_bereinigt'))

# Spalten in Wochentag-Reihenfolge
col_order = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']
pivot_heat = pivot_heat.reindex(columns=[c for c in col_order if c in pivot_heat.columns])

fig = px.imshow(
    pivot_heat,
    title='Durchschnittlicher Verbrauch nach Stunde und Wochentag (kWh)',
    labels=dict(x='Wochentag', y='Stunde', color='kWh'),
    color_continuous_scale='YlOrRd',
    aspect='auto',
    text_auto='.0f',
)
fig.update_layout(height=550)
fig.show()

In [ ]:
# ── Saisonalität: Box-Plot nach Monat ────────────────────────────────────────
df['monat_name_str'] = df['monat'].map(monat_namen)

# Wichtig: Reihenfolge der Monate korrekt setzen
month_order = ['Jan', 'Feb', 'Mär', 'Apr', 'Mai', 'Jun',
               'Jul', 'Aug', 'Sep', 'Okt', 'Nov', 'Dez']

fig = px.box(
    df,
    x='monat_name_str',
    y='verbrauch_bereinigt',
    category_orders={'monat_name_str': month_order},
    title='Verteilung des stündlichen Verbrauchs nach Monat (kWh)',
    labels={'monat_name_str': 'Monat', 'verbrauch_bereinigt': 'Verbrauch (kWh)'},
    color='monat_name_str',
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
fig.update_layout(showlegend=False, height=450)
fig.show()

In [ ]:
# ── Rolling Mean (7-Tage = 168 Stunden) ─────────────────────────────────────
df_daily = df.set_index('timestamp')['verbrauch_bereinigt'].resample('D').mean().reset_index()
df_daily.columns = ['datum', 'verbrauch_daily']
df_daily['rolling_7d'] = df_daily['verbrauch_daily'].rolling(window=7, center=True).mean()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_daily['datum'], y=df_daily['verbrauch_daily'],
    mode='lines', name='Tagesdurchschnitt',
    line=dict(color='lightsteelblue', width=1),
    opacity=0.7
))
fig.add_trace(go.Scatter(
    x=df_daily['datum'], y=df_daily['rolling_7d'],
    mode='lines', name='7-Tage Rolling Mean',
    line=dict(color='steelblue', width=2.5)
))
fig.update_layout(
    title='Täglicher Verbrauch mit 7-Tage-Glättung (kWh)',
    xaxis_title='Datum', yaxis_title='Durchschnittlicher Verbrauch (kWh)',
    height=400
)
fig.show()

In [ ]:
# ── Ausreißer identifizieren (IQR-Methode) ───────────────────────────────────
Q1 = df['verbrauch_bereinigt'].quantile(0.25)
Q3 = df['verbrauch_bereinigt'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df['ist_ausreisser'] = (df['verbrauch_bereinigt'] < lower) | (df['verbrauch_bereinigt'] > upper)
ausreisser = df[df['ist_ausreisser']]

print(f"IQR-Grenzen: [{lower:.1f}, {upper:.1f}] kWh")
print(f"Ausreißer gefunden: {len(ausreisser)} ({len(ausreisser)/len(df)*100:.1f}%)")
print()
print("Ausreißer nach Schicht:")
print(ausreisser['schicht'].value_counts())

# Visualisierung: Ausreißer im Jahreszeitverlauf
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df[~df['ist_ausreisser']]['timestamp'],
    y=df[~df['ist_ausreisser']]['verbrauch_bereinigt'],
    mode='lines', name='Normal',
    line=dict(color='steelblue', width=0.5), opacity=0.5
))
fig.add_trace(go.Scatter(
    x=ausreisser['timestamp'], y=ausreisser['verbrauch_bereinigt'],
    mode='markers', name='Ausreißer',
    marker=dict(color='red', size=5, symbol='x')
))
fig.update_layout(title='Energieverbrauch 2024 – Ausreißer hervorgehoben',
                  xaxis_title='Zeit', yaxis_title='Verbrauch (kWh)', height=400)
fig.show()

In [ ]:
# ── Schicht-Analyse ───────────────────────────────────────────────────────────
schicht_stats = df.groupby('schicht').agg(
    verbrauch_mean=('verbrauch_bereinigt', 'mean'),
    verbrauch_sum=('verbrauch_bereinigt', 'sum'),
    stunden=('verbrauch_bereinigt', 'count')
).reset_index().round(2)
schicht_stats['anteil_pct'] = (schicht_stats['verbrauch_sum'] / schicht_stats['verbrauch_sum'].sum() * 100).round(1)

print("=== Verbrauch nach Schicht ===")
print(schicht_stats.to_string(index=False))

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Ø Verbrauch je Stunde (kWh)', 'Gesamtanteil am Jahresverbrauch (%)'])

farben = {'Früh': 'steelblue', 'Spät': 'coral', 'Nacht': 'lightgrey'}
fig.add_trace(
    go.Bar(x=schicht_stats['schicht'], y=schicht_stats['verbrauch_mean'],
           marker_color=[farben[s] for s in schicht_stats['schicht']],
           text=schicht_stats['verbrauch_mean'].round(1),
           textposition='outside', name='Ø kWh'),
    row=1, col=1
)
fig.add_trace(
    go.Pie(labels=schicht_stats['schicht'],
           values=schicht_stats['anteil_pct'],
           marker_colors=[farben[s] for s in schicht_stats['schicht']],
           name='Anteil'),
    row=1, col=2
)
fig.update_layout(title='Energieverbrauch nach Schicht', height=400, showlegend=False)
fig.show()

# Einsparpotenzial Nachtschicht
nacht_mean = schicht_stats[schicht_stats['schicht'] == 'Nacht']['verbrauch_mean'].values[0]
frueh_mean = schicht_stats[schicht_stats['schicht'] == 'Früh']['verbrauch_mean'].values[0]
nacht_stunden = schicht_stats[schicht_stats['schicht'] == 'Nacht']['stunden'].values[0]
print(f"\nEinsparpotenzial Nachtschicht (auf Früh-Niveau reduzieren):")
print(f"  Nacht-Verbrauch: {nacht_mean:.1f} kWh/h")
print(f"  Früh-Verbrauch:  {frueh_mean:.1f} kWh/h")
print(f"  Einsparung/Jahr: {(frueh_mean - nacht_mean) * nacht_stunden:,.0f} kWh (wenn Nacht = Früh)")
print(f"  Hinweis: Nacht-Verbrauch ist erwartungsgemäß NIEDRIGER – das System arbeitet korrekt.")

## Zusammenfassung & Erkenntnisse

**Technische Erkenntnisse:**

1. **Fehlende Werte (~5%)** waren zufällig verteilt und wurden erfolgreich per linearer Interpolation bereinigt.
2. **Tages-Rhythmus**: Klarer Peak während der Früh- und Spätschicht (Betriebsstunden 6-22 Uhr).
3. **Saisonalität**: Wintermonate zeigen höheren Verbrauch (Heizung, weniger Tageslicht).
4. **Ausreißer**: Mittels IQR-Methode wurden untypische Messwerte identifiziert – zur Prüfung freigegeben.
5. **Wochenend-Effekt**: Signifikant niedrigerer Verbrauch an Samstag/Sonntag.

**Handlungsempfehlungen:**
- Automatische Alarmierung bei Überschreitung des 1,5×IQR-Bereichs einrichten
- Nacht-Standby-Modus für nicht-kritische Anlagen prüfen
- Energieverbrauch mit Produktionsauslastung verknüpfen (weiterer Datensatz nötig)

**DAV-Lernpunkte:**
- `pd.to_datetime()` und `parse_dates` zum korrekten Einlesen von Zeitreihen
- `interpolate(method='linear')` für Lückenfüllung in Zeitreihen
- `.rolling(window).mean()` für gleitende Mittelwerte
- IQR-Methode als robuste Ausreißer-Erkennung

---
*DAV Lab – Prof. Dr. Robert Butscher – THWS Business School*